# Curve risk: a swap book's sensitivities to every quote

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/auto-differentiation/QuantLibAAD-Py/blob/main/notebooks/02_curve_risk.ipynb)

A yield curve is bootstrapped from market quotes, and a book of swaps is priced off it. Its interest-rate risk is
the sensitivity of the book's value to each quote. Bumping needs two bootstraps and book revaluations per quote (central differences).
With QuantLibAAD, the bootstrap and the pricing are recorded once and **one adjoint sweep gives the sensitivity
to every quote**.

The market data and the book are those of the C++
[AdjointTradeRisk example](https://github.com/auto-differentiation/QuantLibAAD/tree/main/Examples/AdjointTradeRisk):
45 quotes and 200 swaps.

In [1]:
%pip install -q QuantLibAAD

## Market quotes

Ten deposits (overnight to 6 months), five FRAs (1x7 to 5x11) and thirty swaps (1 to 30 years) against
6-month Euribor.

In [2]:
import QuantLibAAD as ql
from xad.adj_1st import Tape

today = ql.Date(2, ql.January, 2015)
ql.Settings.instance().evaluationDate = today
calendar = ql.TARGET()
start = calendar.advance(today, 2, ql.Days)

deposit_tenors = [ql.Period(1, ql.Days)] * 3 + [ql.Period(1, ql.Weeks)] + [ql.Period(m, ql.Months) for m in range(1, 7)]
names = (["deposit ON", "deposit TN", "deposit SN", "deposit 1W"] + [f"deposit {m}M" for m in range(1, 7)]
         + [f"FRA {m}x{m + 6}" for m in range(1, 6)] + [f"swap {y}Y" for y in range(1, 31)])
quotes = ([0.0010 + i * 0.0002 for i in range(10)] + [0.0030 + i * 0.0005 for i in range(5)]
          + [0.0060 + i * 0.0001 for i in range(30)])

## The bootstrap

A curve linear in zero rates through one node per quote. AAD differentiates the bootstrap's solver iterations,
so its sensitivities are only as accurate as the solver: we tighten the solver accuracy from its default.

In [3]:
def bootstrap(quotes):
    quote = [ql.QuoteHandle(ql.SimpleQuote(q)) for q in quotes]
    helpers = [
        ql.DepositRateHelper(quote[i], tenor, min(i, 2), calendar, ql.ModifiedFollowing, False, ql.Actual360())
        for i, tenor in enumerate(deposit_tenors)
    ]
    helpers += [
        ql.FraRateHelper(quote[10 + i], i + 1, i + 7, 2, calendar, ql.ModifiedFollowing, False, ql.Actual360())
        for i in range(5)
    ]
    helpers += [
        ql.SwapRateHelper(quote[15 + i], ql.Period(i + 1, ql.Years), calendar, ql.Annual, ql.ModifiedFollowing,
                          ql.Thirty360(ql.Thirty360.European), ql.Euribor6M())
        for i in range(30)
    ]
    curve = ql.PiecewiseLinearZero(today, helpers, ql.Actual365Fixed(), ql.IterativeBootstrap(accuracy=1e-15))
    curve.enableExtrapolation()
    return ql.YieldTermStructureHandle(curve)

## The swap book

200 payer and receiver swaps with random nominals, fixed rates and maturities up to 30 years, from a seeded
random number generator. The book's value is the sum of their NPVs.

In [4]:
rng = ql.MersenneTwisterUniformRsg(1, 42)


def uniform():
    return rng.nextSequence().value()[0].value


book = []
for _ in range(200):
    swap_type = ql.Swap.Payer if uniform() < 0.5 else ql.Swap.Receiver
    nominal = 1e6 * (1 + int(10 * uniform()))
    fixed_rate = 0.002 + 0.008 * uniform()
    years = 1 + int(29.999 * uniform())
    book.append((swap_type, nominal, fixed_rate, years))


def price_swap(trade, curve):
    swap_type, nominal, fixed_rate, years = trade
    end = calendar.advance(start, years, ql.Years)

    def schedule(tenor):
        return ql.Schedule(start, end, tenor, calendar, ql.ModifiedFollowing, ql.ModifiedFollowing,
                           ql.DateGeneration.Backward, False)

    swap = ql.VanillaSwap(swap_type, nominal, schedule(ql.Period(1, ql.Years)), fixed_rate,
                          ql.Thirty360(ql.Thirty360.European), schedule(ql.Period(6, ql.Months)),
                          ql.Euribor6M(curve), 0.0, ql.Actual360())
    swap.setPricingEngine(ql.DiscountingSwapEngine(curve))
    return swap.NPV()


def price_book(quotes):
    curve = bootstrap(quotes)
    return sum(price_swap(trade, curve) for trade in book)


print(f"book value = {price_book(quotes):,.2f}")

book value = -531,879.60


## 1. Book risk in one adjoint sweep

Register the 45 quotes as inputs, record the bootstrap and the pricing of all 200 swaps, and sweep back once from
the book value.

In [5]:
def book_value_and_risk(quotes):
    inputs = [ql.Real(q) for q in quotes]
    with Tape() as tape:
        tape.registerInputs(inputs)
        tape.newRecording()
        value = price_book(inputs)
        tape.registerOutput(value)
        value.derivative = 1.0
        tape.computeAdjoints()
        return value.value, [q.derivative for q in inputs]


value, risk = book_value_and_risk(quotes)

print(f"book value = {value:,.2f}\n")
print("largest sensitivities, per 1bp move of the quote:")
for i in sorted(range(len(risk)), key=lambda i: -abs(risk[i]))[:10]:
    print(f"  {names[i]:12} {risk[i] * 1e-4:14,.2f}")

book value = -531,879.60

largest sensitivities, per 1bp move of the quote:
  swap 14Y         -45,152.25
  swap 21Y         -38,636.67
  swap 28Y         -29,911.59
  swap 22Y         -28,274.69
  swap 25Y          27,279.16
  swap 30Y          26,541.94
  swap 17Y          25,610.10
  swap 7Y           19,043.85
  swap 29Y         -15,346.74
  swap 23Y         -12,606.83


## 2. Check against bumping

Central differences for the five largest sensitivities: two bootstraps and book revaluations per quote.

In [6]:
h = 1e-5
print(f"{'quote':12} {'XAD adjoint':>16} {'bumping':>16} {'rel diff':>10}")
for i in sorted(range(len(risk)), key=lambda i: -abs(risk[i]))[:5]:
    up, down = list(quotes), list(quotes)
    up[i] += h
    down[i] -= h
    bumped = (price_book(up) - price_book(down)) / (2 * h)
    print(f"{names[i]:12} {risk[i]:16,.2f} {bumped:16,.2f} {abs(risk[i] - bumped) / abs(bumped):10.1e}")

quote             XAD adjoint          bumping   rel diff
swap 14Y      -451,522,471.99  -451,522,472.01    4.9e-11
swap 21Y      -386,366,675.21  -386,366,675.25    9.7e-11
swap 28Y      -299,115,874.32  -299,115,874.10    7.4e-10
swap 22Y      -282,746,872.95  -282,746,872.90    1.8e-10
swap 25Y       272,791,566.37   272,791,565.78    2.1e-09


The adjoint sensitivities agree with bumping to its truncation and rounding error. Bumping all 45 quotes would
take 90 bootstraps and book revaluations; the adjoint sweep gives every sensitivity from one recording.

One sweep gives the derivatives of one output. For the risk of each swap separately, register every NPV as an
output and sweep once per swap on the same recording; the C++ example shows a faster way that records the
curve once and each swap separately.

## Next steps

- [Quickstart](https://colab.research.google.com/github/auto-differentiation/QuantLibAAD-Py/blob/main/notebooks/01_quickstart.ipynb): option Greeks in one adjoint sweep.
- [Python examples](https://github.com/auto-differentiation/QuantLibAAD-Py/tree/main/Python/examples) and the [QuantLibAAD documentation](https://auto-differentiation.github.io/quantlib-aad/python/).